# Tree Builder (v2 — Bottom-Up Summary Construction)

**What changed from v1:**
- Old: a single LLM call built the *entire* 3-level tree (root -> chapters -> sections) at once.
- New: the tree is built **bottom-up** in 3 stages so that **every** node (section, chapter, root)
  carries a rich `summary` + `keywords`. This is required by `retreivalPDF.ipynb` v2, which routes
  queries using `keyword_overlap(query, node.summary)` with **zero LLM calls** at query time.

```
Step 1  Group consecutive pages into sections      (LLM, small batches, per chapter)
Step 2  Merge each chapter's section summaries      (LLM, 1 call per chapter)
        into a single chapter summary
Step 3  Merge all chapter summaries into one         (LLM, 1 call)
        root summary
```

Chapter *boundaries* (title + page range) are still detected with a single lightweight LLM call
over the whole page list -- same idea as v1 -- only the bottom-up summary/keyword enrichment is new.

**Run order:** `pageMetadata.ipynb` -> `treeBuilder.ipynb` -> `retreivalPDF.ipynb` -> `answerGeneration.ipynb`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. Imports & Connection

In [2]:
import json
import re

from src.config.db import get_connection
from src.config.llm import llm

conn = get_connection()

## 2. Set Document ID
Change `document_id` to the document you want to build a tree for.

In [3]:
document_id = "DOC000001"

## 3. Fetch Pages with Metadata
Pulls every page for the document, including the `metadata` JSON column produced by
`pageMetadata.ipynb`. If a page has no metadata, its title/summary/keywords default to empty.

In [4]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            id,
            "pageNumber",
            content,
            metadata
        FROM "Page"
        WHERE "documentId" = %s
        ORDER BY "pageNumber"
        """,
        (document_id,),
    )
    rows = cur.fetchall()

pages = []
for row in rows:
    meta = row[3] if row[3] else {}
    pages.append({
        "id":         row[0],
        "pageNumber": row[1],
        "content":    row[2],
        "title":      meta.get("title", ""),
        "summary":    meta.get("summary", ""),
        "keywords":   meta.get("keywords", []),
        "topics":     meta.get("topics", []),
        "pageType":   meta.get("pageType", "content"),
    })

print(f"Fetched {len(pages)} pages")
pages[:2]

Fetched 50 pages


[{'id': 'DOC000001_P001',
  'pageNumber': 1,
  'content': 'REPORT TO CONGRESS\n110th\nAnnual Repor t of the Board of\nGovernors of the F ederal Reser ve System\n2023\nBOARD OF GO VERNORS OF THE FEDERAL RESER VE SYSTEM',
  'title': 'REPORT TO CONGRESS',
  'summary': '110th Annual Report of the Board of Governors of the Federal Reserve System',
  'keywords': ['Federal Reserve', 'Annual Report', 'Board of Governors'],
  'topics': ['Economy', 'Monetary Policy', 'Financial Stability'],
  'pageType': 'cover'},
 {'id': 'DOC000001_P002',
  'pageNumber': 2,
  'content': '',
  'title': '',
  'summary': '',
  'keywords': [],
  'topics': [],
  'pageType': 'content'}]

## 4. Step 1a -- Detect Chapter Boundaries
**One lightweight LLM call** over titles/summaries of every page (no full page text) to identify
chapters and their page ranges. Sections are *not* decided here -- that happens per-chapter in
small batches next, which keeps each individual LLM call small even for very long documents.

In [5]:
CHAPTER_PROMPT = """
You are identifying the top-level CHAPTER structure of a document for a Vectorless RAG system.

Group the pages below into chapters (major topic groups). Every page must fall inside exactly
one chapter's page range. Chapters must be in page order and cover the full range with no gaps.
Chapter titles must be descriptive (not "Chapter 1").

Return ONLY valid JSON. No markdown, no explanation.

Schema:
[
  {{"title": "<chapter title>", "pageStart": <int>, "pageEnd": <int>}}
]

Pages (pageNumber | title | summary):
{pages}
"""


def format_pages_for_prompt(pages):
    lines = []
    for p in pages:
        summary_short = (p["summary"] or "")[:200].replace("\n", " ")
        lines.append(
            f"Page {p['pageNumber']} | {p['title'] or '(no title)'} | {summary_short}"
        )
    return "\n".join(lines)


def call_llm_for_chapters(pages):
    prompt = CHAPTER_PROMPT.format(pages=format_pages_for_prompt(pages))
    response = llm.invoke(prompt)
    text = response.content.strip()

    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    return json.loads(text.strip())


chapters = call_llm_for_chapters(pages)
print(f"Detected {len(chapters)} chapters")
for ch in chapters:
    print(f"  {ch['title']}  (pages {ch['pageStart']}-{ch['pageEnd']})")

Detected 5 chapters
  Introduction and Overview  (pages 1-6)
  Monetary Policy and Economic Developments  (pages 7-20)
  Financial Stability  (pages 21-30)
  Supervision and Regulation  (pages 31-44)
  International Engagement and Financial Stability  (pages 45-50)


## 5. Step 1b -- Group Pages into Sections per Chapter
For each chapter, its pages are split into **small batches** (`SECTION_BATCH_SIZE`) and sent to
the LLM one batch at a time. The LLM groups consecutive same-topic pages into sections and
returns an already-merged `summary` + `keywords` per section -- this is the raw material Step 2
merges upward into the chapter summary.

In [6]:
SECTION_BATCH_SIZE = 6

SECTION_PROMPT = """
You are grouping pages from ONE chapter of a document into SECTIONS for a Vectorless RAG system.

Chapter: "{chapter_title}"

Rules:
- Every page below must appear in exactly one section.
- A section spans one or more CONSECUTIVE pages on the same sub-topic.
- Section titles must be descriptive (not "Section 1").
- "summary" must be a 1-2 sentence summary MERGED from the page summaries in that section
  (do not just copy one page's summary).
- "keywords" must be a short deduplicated list merged from the pages' keywords.
- pageIds must be the exact id strings provided -- do not invent new ones.

Return ONLY valid JSON. No markdown, no explanation.

Schema:
[
  {{
    "title": "<section title>",
    "summary": "<merged summary>",
    "keywords": ["..."],
    "pageStart": <int>,
    "pageEnd": <int>,
    "pageIds": ["<page_id>", ...]
  }}
]

Pages (pageNumber | page_id | title | summary | keywords):
{pages}
"""


def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def format_pages_for_sections(pages_batch):
    lines = []
    for p in pages_batch:
        summary_short = (p["summary"] or "")[:200].replace("\n", " ")
        kw = ", ".join(p["keywords"] or [])
        lines.append(
            f"Page {p['pageNumber']} | {p['id']} | {p['title'] or '(no title)'} | {summary_short} | {kw}"
        )
    return "\n".join(lines)


def call_llm_for_sections(chapter_title, pages_batch):
    prompt = SECTION_PROMPT.format(
        chapter_title=chapter_title,
        pages=format_pages_for_sections(pages_batch),
    )
    response = llm.invoke(prompt)
    text = response.content.strip()

    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    return json.loads(text.strip())


def generate_sections_for_chapter(chapter, all_pages):
    chapter_pages = [
        p for p in all_pages
        if chapter["pageStart"] <= p["pageNumber"] <= chapter["pageEnd"]
    ]

    sections = []
    for batch in batches(chapter_pages, SECTION_BATCH_SIZE):
        sections.extend(call_llm_for_sections(chapter["title"], batch))
    return sections


for chapter in chapters:
    chapter["sections"] = generate_sections_for_chapter(chapter, pages)
    print(f"[{chapter['title']}] -> {len(chapter['sections'])} sections")

[Introduction and Overview] -> 5 sections
[Monetary Policy and Economic Developments] -> 8 sections
[Financial Stability] -> 5 sections
[Supervision and Regulation] -> 8 sections
[International Engagement and Financial Stability] -> 4 sections


## 6. Step 2 -- Merge Section Summaries -> Chapter Summary
One LLM call **per chapter**, using only the (already short) section summaries/keywords produced
in Step 1 -- never the raw page text. Cheap even for chapters with many sections.

In [7]:
CHAPTER_SUMMARY_PROMPT = """
You are writing a single merged summary for a document CHAPTER, based on the summaries of its
sections, for a Vectorless RAG system.

Chapter title: "{chapter_title}"

Section summaries:
{sections}

Return ONLY valid JSON. No markdown, no explanation.

Schema:
{{"summary": "<2-3 sentence merged summary covering the whole chapter>", "keywords": ["..."]}}
"""


def format_sections_for_prompt(sections):
    lines = []
    for s in sections:
        kw = ", ".join(s.get("keywords", []) or [])
        lines.append(f"- {s['title']}: {s['summary']} (keywords: {kw})")
    return "\n".join(lines)


def merge_chapter_summary(chapter):
    prompt = CHAPTER_SUMMARY_PROMPT.format(
        chapter_title=chapter["title"],
        sections=format_sections_for_prompt(chapter["sections"]),
    )
    response = llm.invoke(prompt)
    text = response.content.strip()

    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    return json.loads(text.strip())


for chapter in chapters:
    merged = merge_chapter_summary(chapter)
    chapter["summary"] = merged.get("summary", "")
    chapter["keywords"] = merged.get("keywords", [])
    print(f"[{chapter['title']}] summary: {chapter['summary'][:100]}...")

[Introduction and Overview] summary: This chapter provides an introduction and overview of the Federal Reserve System, including its 110t...
[Monetary Policy and Economic Developments] summary: The Federal Reserve conducted monetary policy operations in 2023 to promote maximum employment, stab...
[Financial Stability] summary: The Federal Reserve plays a crucial role in ensuring financial stability by monitoring the financial...
[Supervision and Regulation] summary: The Federal Reserve promotes a safe and sound banking and financial system through supervision and r...
[International Engagement and Financial Stability] summary: The chapter discusses the Federal Reserve's international engagement and efforts to promote financia...


## 7. Step 3 -- Merge Chapter Summaries -> Root Summary
The final LLM call of the pipeline -- merges all chapter-level summaries into one root summary
covering the whole document. `retreivalPDF.ipynb` doesn't route off the root node directly today,
but downstream consumers (evaluation, UI previews) benefit from a document-level summary too.

In [8]:
ROOT_SUMMARY_PROMPT = """
You are writing a single merged summary for an ENTIRE document, based on the summaries of its
chapters, for a Vectorless RAG system.

Document title: "{doc_title}"

Chapter summaries:
{chapters}

Return ONLY valid JSON. No markdown, no explanation.

Schema:
{{"summary": "<3-4 sentence merged summary covering the whole document>", "keywords": ["..."]}}
"""


def format_chapters_for_prompt(chapters):
    lines = []
    for c in chapters:
        kw = ", ".join(c.get("keywords", []) or [])
        lines.append(f"- {c['title']}: {c['summary']} (keywords: {kw})")
    return "\n".join(lines)


with conn.cursor() as cur:
    cur.execute(
        'SELECT title FROM "Document" WHERE id = %s',
        (document_id,),
    )
    doc_row = cur.fetchone()
doc_title = doc_row[0] if doc_row else "Document"


def merge_root_summary(doc_title, chapters):
    prompt = ROOT_SUMMARY_PROMPT.format(
        doc_title=doc_title,
        chapters=format_chapters_for_prompt(chapters),
    )
    response = llm.invoke(prompt)
    text = response.content.strip()

    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]

    return json.loads(text.strip())


root_merged = merge_root_summary(doc_title, chapters)
root_summary = root_merged.get("summary", "")
root_keywords = root_merged.get("keywords", [])

print("Root summary:", root_summary)
print("Root keywords:", root_keywords)

Root summary: The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including monetary policy operations and financial stability efforts. The report highlights the Federal Reserve's role in promoting maximum employment, stable prices, and moderate long-term interest rates, as well as its efforts to ensure financial stability and regulate the banking and financial system. The Federal Reserve's international engagement and cooperation with other regulatory bodies are also discussed, including its role in the Financial Stability Board and compliance with anti-money laundering laws. The report ultimately aims to promote a safe and sound banking and financial system, supporting the stability of the U.S. economy.
Root keywords: ['Federal Reserve', 'Annual Report', 'Monetary Policy', 'Financial Stability', 'Banking System', 'Regulation', 'International Engagement', 'Financial Stability Board', 'Compliance', 'Risk Management']


## 8. Assemble the Full Tree
Combine root + chapters + sections into the same nested JSON shape v1 produced, but now with `summary`/`keywords` at every level.

In [9]:
def assemble_tree(doc_title, root_summary, root_keywords, chapters):
    tree = {
        "title": doc_title,
        "type": "root",
        "summary": root_summary,
        "keywords": root_keywords,
        "children": [],
    }

    for chapter in chapters:
        chapter_node = {
            "title": chapter["title"],
            "type": "chapter",
            "summary": chapter.get("summary", ""),
            "keywords": chapter.get("keywords", []),
            "children": [],
        }
        for section in chapter["sections"]:
            chapter_node["children"].append({
                "title": section["title"],
                "type": "section",
                "summary": section.get("summary", ""),
                "keywords": section.get("keywords", []),
                "pageStart": section["pageStart"],
                "pageEnd": section["pageEnd"],
                "pageIds": section["pageIds"],
            })
        tree["children"].append(chapter_node)

    return tree


raw_tree = assemble_tree(doc_title, root_summary, root_keywords, chapters)
print(json.dumps(raw_tree, indent=2)[:1500])

{
  "title": "2023-annual-report-truncated",
  "type": "root",
  "summary": "The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including monetary policy operations and financial stability efforts. The report highlights the Federal Reserve's role in promoting maximum employment, stable prices, and moderate long-term interest rates, as well as its efforts to ensure financial stability and regulate the banking and financial system. The Federal Reserve's international engagement and cooperation with other regulatory bodies are also discussed, including its role in the Financial Stability Board and compliance with anti-money laundering laws. The report ultimately aims to promote a safe and sound banking and financial system, supporting the stability of the U.S. economy.",
  "keywords": [
    "Federal Reserve",
    "Annual Report",
    "Monetary Policy",
    "Financial Stability",
    "Banking System",
    "Regulation",
    "Internation

## 9. Validate & Enrich Tree
- Checks every `pageId` from the DB is assigned to a section.
- Derives `pageStart`, `pageEnd`, `pageCount` for chapters/root from their children.
- Assigns stable `path` strings (`root`, `1`, `1.1`, `1.2`, ...) for traversal.
- Leaves `summary`/`keywords` untouched -- they were already set in Step 1-3.

In [10]:
def enrich_and_validate(tree: dict, all_pages: list) -> dict:
    all_page_ids   = {p["id"] for p in all_pages}
    page_id_to_num = {p["id"]: p["pageNumber"] for p in all_pages}
    seen_ids: set  = set()

    chapter_idx = 0
    for chapter in tree.get("children", []):
        chapter_idx += 1
        chapter["level"] = 1
        chapter["path"]  = str(chapter_idx)

        section_idx        = 0
        chapter_page_count = 0
        chapter_page_start = None
        chapter_page_end   = None

        for section in chapter.get("children", []):
            section_idx += 1
            section["level"] = 2
            section["path"]  = f"{chapter_idx}.{section_idx}"

            s_ids  = section.get("pageIds", [])
            s_nums = sorted(
                [page_id_to_num[pid] for pid in s_ids if pid in page_id_to_num]
            )

            if s_nums:
                section["pageStart"] = s_nums[0]
                section["pageEnd"]   = s_nums[-1]
            section["pageCount"] = len(s_ids)

            seen_ids.update(s_ids)
            chapter_page_count += len(s_ids)

            if s_nums:
                if chapter_page_start is None or s_nums[0] < chapter_page_start:
                    chapter_page_start = s_nums[0]
                if chapter_page_end is None or s_nums[-1] > chapter_page_end:
                    chapter_page_end = s_nums[-1]

        chapter["pageStart"] = chapter_page_start
        chapter["pageEnd"]   = chapter_page_end
        chapter["pageCount"] = chapter_page_count

    tree["level"]     = 0
    tree["path"]      = "root"
    tree["pageCount"] = len(all_pages)
    tree["pageStart"] = min(p["pageNumber"] for p in all_pages)
    tree["pageEnd"]   = max(p["pageNumber"] for p in all_pages)

    missing = all_page_ids - seen_ids
    if missing:
        missing_nums = sorted(
            [page_id_to_num[pid] for pid in missing if pid in page_id_to_num]
        )
        print(f"WARNING: {len(missing)} pages not assigned to any section: pages {missing_nums}")
    else:
        print("All pages accounted for.")

    return tree


enriched_tree = enrich_and_validate(raw_tree, pages)
print(json.dumps(enriched_tree, indent=2)[:1500])

All pages accounted for.
{
  "title": "2023-annual-report-truncated",
  "type": "root",
  "summary": "The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including monetary policy operations and financial stability efforts. The report highlights the Federal Reserve's role in promoting maximum employment, stable prices, and moderate long-term interest rates, as well as its efforts to ensure financial stability and regulate the banking and financial system. The Federal Reserve's international engagement and cooperation with other regulatory bodies are also discussed, including its role in the Financial Stability Board and compliance with anti-money laundering laws. The report ultimately aims to promote a safe and sound banking and financial system, supporting the stability of the U.S. economy.",
  "keywords": [
    "Federal Reserve",
    "Annual Report",
    "Monetary Policy",
    "Financial Stability",
    "Banking System",
    "Regu

In [ ]:
"""
Cell(s) to ADD to treeBuilder.ipynb, AFTER `enriched_tree = enrich_and_validate(...)`
(step 9) and BEFORE `upsert_tree(...)` (step 11). Uses the same `conn`/`document_id`
already open in that notebook.
 
Pipeline this implements:
    Page metadata -> Build section candidates -> Create base tree
      -> Map tables to tree nodes -> Map figures to tree nodes -> Final enriched tree
 
No LLM calls needed for the mapping itself: every section already carries an exact
pageStart/pageEnd range from step 9, and every table/figure has an exact pageNumber
(from the Page it belongs to) -- so the mapping is a deterministic range lookup,
which is both cheaper and more reliable than asking an LLM to guess.
"""

In [ ]:
import json
def load_tables_with_pages(conn, document_id: str) -> list:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT t.id, t."tableNumber", t.title, t.caption, p."pageNumber"
            FROM "Table" t
            JOIN "Page" p ON p.id = t."pageId"
            WHERE t."documentId" = %s
            ''',
            (document_id,),
        )
        rows = cur.fetchall()
    return [
        {"id": r[0], "tableNumber": r[1], "title": r[2], "caption": r[3], "pageNumber": r[4]}
        for r in rows
    ]

In [ ]:
def load_figures_with_pages(conn, document_id: str) -> list:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT f.id, f."figureNumber", f.title, f.caption, p."pageNumber"
            FROM "Figure" f
            JOIN "Page" p ON p.id = f."pageId"
            WHERE f."documentId" = %s
            ''',
            (document_id,),
        )
        rows = cur.fetchall()
    return [
        {"id": r[0], "figureNumber": r[1], "title": r[2], "caption": r[3], "pageNumber": r[4]}
        for r in rows
    ]

In [ ]:
def map_items_to_tree(tree: dict, tables: list, figures: list) -> dict:
    """Walk to every section (leaf) node and attach the tables/figures whose
    pageNumber falls inside that section's [pageStart, pageEnd] range."""
 
    def in_range(item, p_start, p_end):
        return p_start is not None and p_end is not None and p_start <= item["pageNumber"] <= p_end
 
    def assign(node):
        if node.get("type") == "section":
            p_start, p_end = node.get("pageStart"), node.get("pageEnd")
 
            matched_tables = [t for t in tables if in_range(t, p_start, p_end)]
            matched_figures = [f for f in figures if in_range(f, p_start, p_end)]
 
            node["tableIds"] = [t["id"] for t in matched_tables]
            node["figureIds"] = [f["id"] for f in matched_figures]
 
            # lightweight summaries kept ON the tree so retrieval can route on them
            # without a DB round trip
            node["tableSummaries"] = [
                {"id": t["id"], "title": t["title"] or t["tableNumber"], "caption": t["caption"] or ""}
                for t in matched_tables
            ]
            node["figureSummaries"] = [
                {"id": f["id"], "title": f["title"] or f["figureNumber"], "caption": f["caption"] or ""}
                for f in matched_figures
            ]
        for child in node.get("children", []):
            assign(child)
 
    assign(tree)
    return tree
 
 
tables_for_mapping = load_tables_with_pages(conn, document_id)
figures_for_mapping = load_figures_with_pages(conn, document_id)
 
enriched_tree = map_items_to_tree(enriched_tree, tables_for_mapping, figures_for_mapping)
 
total_tables = sum(len(s.get("tableIds", [])) for ch in enriched_tree["children"] for s in ch["children"])
total_figures = sum(len(s.get("figureIds", [])) for ch in enriched_tree["children"] for s in ch["children"])
print(f"Mapped {total_tables} table refs and {total_figures} figure refs onto tree sections")
 
# `enriched_tree` now flows into step 11 (`upsert_tree(conn, document_id, enriched_tree)`)
# exactly as before -- no other changes needed to that cell.
 

## 10. In-Memory Tree Nodes & Traversal Demo
Same lightweight `TreeNode` wrapper used by `retreivalPDF.ipynb` -- every node now exposes `.summary` and `.keywords`.

In [11]:
class TreeNode:
    """Lightweight wrapper around a tree dict node."""

    def __init__(self, data: dict, parent=None):
        self.data     = data
        self.parent   = parent
        self.children = []

    @property
    def level(self):    return self.data.get("level", 0)
    @property
    def title(self):    return self.data.get("title", "")
    @property
    def summary(self):  return self.data.get("summary", "")
    @property
    def keywords(self): return self.data.get("keywords", [])
    @property
    def path(self):     return self.data.get("path", "")
    @property
    def type(self):     return self.data.get("type", "root")

    def __repr__(self):
        return f"TreeNode(level={self.level}, path={self.path!r}, title={self.title!r})"


def build_tree_nodes(tree_dict: dict, parent=None) -> TreeNode:
    node = TreeNode(tree_dict, parent)
    for child_dict in tree_dict.get("children", []):
        node.children.append(build_tree_nodes(child_dict, parent=node))
    return node


tree_root = build_tree_nodes(enriched_tree)

print("Root :", tree_root)
print(f"Root summary: {tree_root.summary[:150]}...")
print(f"Chapters: {len(tree_root.children)}")
for ch in tree_root.children:
    print(f"  [{ch.path}] {ch.title}  (pages {ch.data.get('pageStart')}-{ch.data.get('pageEnd')})")
    print(f"      summary: {ch.summary[:100]}...")
    for sec in ch.children:
        print(f"    [{sec.path}] {sec.title}  ({sec.data.get('pageCount')} pages)")

Root : TreeNode(level=0, path='root', title='2023-annual-report-truncated')
Root summary: The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including monetary policy operations and fin...
Chapters: 5
  [1] Introduction and Overview  (pages 1-6)
      summary: This chapter provides an introduction and overview of the Federal Reserve System, including its 110t...
    [1.1] Report and Contents  (3 pages)
    [1.2] Appendixes  (1 pages)
    [1.3] Introduction to the Federal Reserve  (1 pages)
    [1.4] Untitled Page  (1 pages)
    [1.5] Untitled Page  (1 pages)
  [2] Monetary Policy and Economic Developments  (pages 7-20)
      summary: The Federal Reserve conducted monetary policy operations in 2023 to promote maximum employment, stab...
    [2.1] Introduction to Federal Reserve Operations  (2 pages)
    [2.2] Monetary Policy and Economic Developments Overview  (2 pages)
    [2.3] Economic Activity and Financial Stability  (2 pages)


## 11. Persist Tree to PostgreSQL
Upserts into the `Tree` table (unique on `documentId`). If a tree already exists for this
document, `treeJson` is overwritten and `version` is bumped. Unchanged from v1.

In [12]:
def upsert_tree(conn, document_id: str, tree_dict: dict):
    tree_json_str = json.dumps(tree_dict)

    with conn.cursor() as cur:
        cur.execute(
            'SELECT id, version FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        existing = cur.fetchone()

        if existing:
            tree_id, current_version = existing
            new_version = current_version + 1
            cur.execute(
                """
                UPDATE "Tree"
                SET
                    "treeJson"  = %s,
                    version     = %s,
                    "createdAt" = NOW()
                WHERE id = %s
                """,
                (tree_json_str, new_version, tree_id),
            )
            print(f"Updated existing tree (id={tree_id}) -> version {new_version}")
        else:
            cur.execute(
                """
                INSERT INTO "Tree"
                    (id, "documentId", "treeJson", version)
                VALUES
                    (gen_random_uuid()::text, %s, %s, 1)
                """,
                (document_id, tree_json_str),
            )
            print(f"Inserted new tree for document {document_id}")

    conn.commit()
    print("Tree stored successfully.")


upsert_tree(conn, document_id, enriched_tree)

Updated existing tree (id=76602173-f929-45bd-89df-0df28878ed3b) -> version 2
Tree stored successfully.


## 12. Verify -- Read Back from DB

In [13]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, "documentId", version, "treeJson"
        FROM "Tree"
        WHERE "documentId" = %s
        """,
        (document_id,),
    )
    row = cur.fetchone()

if row:
    tree_id, doc_id, version, stored_json = row
    if isinstance(stored_json, str):
        stored_json = json.loads(stored_json)

    print(f"tree.id      = {tree_id}")
    print(f"documentId   = {doc_id}")
    print(f"version      = {version}")
    print(f"root title   = {stored_json.get('title')}")
    print(f"root summary = {stored_json.get('summary', '')[:120]}...")
    print(f"chapters     = {len(stored_json.get('children', []))}")
    total_sections = sum(
        len(ch.get("children", []))
        for ch in stored_json.get("children", [])
    )
    print(f"sections     = {total_sections}")
    print(f"total pages  = {stored_json.get('pageCount')}")
else:
    print("No tree found for this document.")

tree.id      = 76602173-f929-45bd-89df-0df28878ed3b
documentId   = DOC000001
version      = 2
root title   = 2023-annual-report-truncated
root summary = The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including mone...
chapters     = 5
sections     = 30
total pages  = 50


## 13. Reload Helper -- Use in Other Notebooks
`load_tree_from_db` is copied into `retreivalPDF.ipynb` / `answerGeneration.ipynb` so each notebook
can run standalone. Because every node now carries `summary` + `keywords`, those notebooks can
route queries with `keyword_overlap(query, node.summary)` and **0 LLM calls**.

In [14]:
def load_tree_from_db(conn, document_id: str) -> TreeNode:
    """Load treeJson from DB and return the root TreeNode."""
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "treeJson" FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        row = cur.fetchone()

    if row is None:
        raise ValueError(f"No tree found for documentId={document_id!r}")

    tree_dict = row[0]
    if isinstance(tree_dict, str):
        tree_dict = json.loads(tree_dict)

    return build_tree_nodes(tree_dict)


reloaded_root = load_tree_from_db(conn, document_id)
print("Reloaded:", reloaded_root)
print(f"Root summary: {reloaded_root.summary[:150]}...")
for ch in reloaded_root.children:
    print(f"  [{ch.path}] {ch.title}  pages {ch.data['pageStart']}-{ch.data['pageEnd']}")
    for sec in ch.children:
        print(f"    [{sec.path}] {sec.title}  ({sec.data['pageCount']} pages)")

Reloaded: TreeNode(level=0, path='root', title='2023-annual-report-truncated')
Root summary: The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including monetary policy operations and fin...
  [1] Introduction and Overview  pages 1-6
    [1.1] Report and Contents  (3 pages)
    [1.2] Appendixes  (1 pages)
    [1.3] Introduction to the Federal Reserve  (1 pages)
    [1.4] Untitled Page  (1 pages)
    [1.5] Untitled Page  (1 pages)
  [2] Monetary Policy and Economic Developments  pages 7-20
    [2.1] Introduction to Federal Reserve Operations  (2 pages)
    [2.2] Monetary Policy and Economic Developments Overview  (2 pages)
    [2.3] Economic Activity and Financial Stability  (2 pages)
    [2.4] Monetary Policy Overview  (1 pages)
    [2.5] Labor Market and Earnings  (3 pages)
    [2.6] International Developments and Financial Stability  (1 pages)
    [2.7] Monetary Policy and Economic Developments  (1 pages)
    [2.8] Monetary Poli